# Step 3 — Predict & Visualise Channel Unmixing

This notebook runs MicroSplit prediction on the small cpg0000 sample
and visualises channel unmixing results.

**Prerequisites:** Run notebooks 00–02 first.

For large-scale plate-level prediction on HPC, see `../cpg0000-jump-pilot/predict.sh`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import tifffile

sys.path.insert(0, '../../../src')  # point at JUMP-MicroSplit/src

DATASET_DIR  = Path('./cpg0000_training_dataset')
OUTPUT_DIR   = Path('./cpg0000_predictions')
CHANNELS     = ['DNA', 'RNA', 'ER', 'AGP', 'Mito']

# cpg0000 channel mapping
CHANNEL_MAPPING = {'DNA': 5, 'RNA': 3, 'ER': 4, 'AGP': 2, 'Mito': 1}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATASET_DIR.exists():
    raise FileNotFoundError(f'Dataset not found at {DATASET_DIR}. Run earlier notebooks.')

## Load model and stats

In [ ]:
from microsplit_reproducibility.workflows.cellpainting import (
    find_checkpoint,
    load_model_and_stats,
)

checkpoint = find_checkpoint(str(DATASET_DIR))
model, stats = load_model_and_stats(
    training_dir=str(DATASET_DIR),
    checkpoint_path=checkpoint,
    channel_names=CHANNELS,
)

print(f'Checkpoint: {checkpoint}')
print(f'Stats keys: {list(stats.keys())}')
device = next(model.parameters()).device
print(f'Model device: {device}')

## Load an image from the training dataset and predict

In [ ]:
from microsplit_reproducibility.workflows.cellpainting import predict_fov, compute_metrics

# Load image_id=0 directly from the dataset directory
image_id = 0
channel_images = {
    ch: tifffile.imread(str(DATASET_DIR / ch / f'{image_id:06d}.tiff'))
    for ch in CHANNELS
}

print(f'Input image shapes: {[(k, v.shape) for k, v in channel_images.items()]}')
print(f'Running prediction (MMSE=10, 1 posterior sample)...')

mmse_pred, posterior_samples = predict_fov(
    model=model,
    channel_images=channel_images,
    stats=stats,
    channel_names=CHANNELS,
    image_size=64,
    grid_size=8,       # small grid for notebook speed
    multiscale_lowres_count=3,
    mmse_count=10,     # fewer samples for demo speed
    num_posterior_samples=1,
    posterior_seeds=(42,),
    batch_size=16,
)

print(f'MMSE prediction shape: {mmse_pred.shape}   (H, W, C)')
print(f'Posterior samples: {len(posterior_samples)}')

## Visualise: input combined vs predicted channels

In [ ]:
# Show combined input + ground-truth per channel + MicroSplit prediction
combined = tifffile.imread(str(DATASET_DIR / 'combined' / f'{image_id:06d}.tiff'))

n_cols = 1 + len(CHANNELS)  # combined + N channels
fig, axes = plt.subplots(3, n_cols, figsize=(4 * n_cols, 12))

# Row 0: input combined (repeated for alignment)
for ax in axes[0]:
    ax.imshow(combined, cmap='viridis', vmin=0, vmax=np.percentile(combined, 99.5))
    ax.axis('off')
axes[0, 0].set_title('Combined input', fontsize=10)
for ax, ch in zip(axes[0, 1:], CHANNELS):
    ax.set_title(f'Input combined\n(for {ch})', fontsize=10)

# Row 1: ground truth per channel
axes[1, 0].axis('off')
axes[1, 0].text(0.5, 0.5, 'Ground\ntruth', ha='center', va='center', fontsize=12)
for ax, ch in zip(axes[1, 1:], CHANNELS):
    img = channel_images[ch].astype(float)
    ax.imshow(img, cmap='gray', vmin=0, vmax=np.percentile(img, 99.5))
    ax.set_title(f'GT {ch}', fontsize=10)
    ax.axis('off')

# Row 2: MicroSplit MMSE prediction
axes[2, 0].axis('off')
axes[2, 0].text(0.5, 0.5, 'MicroSplit\nMMSE', ha='center', va='center', fontsize=12)
for c_idx, (ax, ch) in enumerate(zip(axes[2, 1:], CHANNELS)):
    pred = mmse_pred[:, :, c_idx]
    vmax = np.percentile(channel_images[ch], 99.5) if channel_images[ch].max() > 0 else 1
    ax.imshow(pred, cmap='gray', vmin=0, vmax=vmax)
    ax.set_title(f'Pred {ch}', fontsize=10)
    ax.axis('off')

plt.suptitle(f'MicroSplit: combined → individual channels (image_id={image_id})', fontsize=13)
plt.tight_layout()
plt.show()

## Compute PSNR/SSIM metrics

In [ ]:
import pandas as pd
from microsplit_reproducibility.workflows.cellpainting import compute_metrics

metrics = compute_metrics(channel_images, mmse_pred, CHANNELS)

df = pd.DataFrame([
    {'channel': ch, 'PSNR': metrics[f'psnr_{ch}'], 'SSIM': metrics[f'ssim_{ch}']}
    for ch in CHANNELS
])
print('\nMetrics (note: only 5 epochs training, expect lower quality than full runs):')
display(df.to_string(index=False))

print(f"\nMean PSNR: {df['PSNR'].mean():.2f} dB")
print(f"Mean SSIM: {df['SSIM'].mean():.4f}")

## Save predictions

In [ ]:
from microsplit_reproducibility.workflows.cellpainting import save_fov_predictions

save_fov_predictions(
    output_dir=str(OUTPUT_DIR),
    well='demo',
    site=1,
    mmse_prediction=mmse_pred,
    posterior_samples=posterior_samples,
    channel_names=CHANNELS,
    save_uint16=True,
)

saved = list(OUTPUT_DIR.rglob('*.tif'))
print(f'Saved {len(saved)} TIFF files to {OUTPUT_DIR}')
for f in sorted(saved)[:10]:
    print(f'  {f.relative_to(OUTPUT_DIR)}')

## Visualise posterior uncertainty

In [ ]:
# Show MMSE vs posterior sample for one channel
ch_idx = 0  # DNA
ch = CHANNELS[ch_idx]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

gt = channel_images[ch].astype(float)
vmax = np.percentile(gt, 99.5) if gt.max() > 0 else 1

axes[0].imshow(gt, cmap='gray', vmin=0, vmax=vmax)
axes[0].set_title(f'Ground truth {ch}', fontsize=11)

axes[1].imshow(mmse_pred[:, :, ch_idx], cmap='gray', vmin=0, vmax=vmax)
axes[1].set_title(f'MMSE (mean of 10 samples)', fontsize=11)

if posterior_samples:
    axes[2].imshow(posterior_samples[0][:, :, ch_idx], cmap='gray', vmin=0, vmax=vmax)
    axes[2].set_title(f'Posterior sample (seed=42)', fontsize=11)
else:
    axes[2].text(0.5, 0.5, 'No posterior', ha='center', va='center')

for ax in axes:
    ax.axis('off')

plt.suptitle(f'Channel: {ch} — uncertainty quantification', fontsize=13)
plt.tight_layout()
plt.show()

## Summary

You have run the full MicroSplit workflow:

| Step | Notebook | HPC equivalent |
|------|----------|----------------|
| 0 — Download & build dataset | `00_datasets.ipynb` | `cpg0000-jump-pilot/datasets.sh` |
| 1 — Train noise models | `01_noisemodels.ipynb` | `cpg0000-jump-pilot/noise_models.sh` |
| 2 — Train MicroSplit | `02_train.ipynb` | `cpg0000-jump-pilot/train.sh` |
| 3 — Predict & evaluate | `03_predict.ipynb` | `cpg0000-jump-pilot/predict.sh` |

For production runs on the full JUMP-CP datasets, use the bash scripts in the
`cpg000X-*/` experiment folders.